Fitting Data

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from astropy.table import QTable
from astropy import units as u
from astropy import constants as const
from scipy.optimize import curve_fit


Power on the Moon

The Apollo lunar mission deployed a series of experiments on the Moon.
The experiment package was called the Apollo Lunar Surface Experiments Package
(ALSEP)
The ALSEP was powered by a radioisotope thermoelectric generator
(RTG)
An RTG is basically a fist-sized slug of Pu-238 wrapped in a material that generates electric power when heated.
Since the RTG is powered by a radioisotope, the output power decreases over time as the radioisotope decays.

Read in the datafile

The data file
https://uwashington-astro300.github.io/A300_Data/Apollo_RTG.csv
contains the power output of the Apollo 12 RTG as a function of time.
The data colunms are
[Day] - Days on the Moon
[Power] - RTG power output in Watts
Read in the datafile as a astropy
QTable

In [ ]:
my_data1 = QTable.read('https://uwashington-astro300.github.io/A300_Data/Apollo_RTG.csv', 
                       format='ascii.csv')


Add units to the columns

In [ ]:
my_data1['Day'].unit = u.d
my_data1['Power'].unit = u.watt


Plot the Data

Day vs. Power
Fit the function with a (degree = 3) polynomial
Plot the fit with the data
Output size w:11in, h:8.5in
Make the plot look nice (including clear labels)

In [ ]:
my_x = my_data1['Day']
my_y = my_data1['Power']

my_fit1 = np.polyfit(my_x.value, my_y.value, 3) 
fitted_polynomial = np.poly1d(my_fit1)
my_data1['Fit'] = fitted_polynomial(my_x.value)* my_y.unit

fig, ax = plt.subplots(
    figsize = (11, 8.5), 
    constrained_layout = True
)

ax.set_xlabel(f"Day on the Moon [{my_x.unit}]")
ax.set_ylabel(f"RTG power output [{my_y.unit}]")

ax.plot(my_x, my_y,
        color = "m",
        marker = "o",
        linestyle = "None",
        markersize = 10,
        label="Data");

ax.plot(my_x, my_data1['Fit'],
        marker = "None",
        linewidth = 6,
        color = (0.3, 0.1, 0.9, 0.4),
        linestyle = '--',
        label = "Fit to Data")

ax.legend(loc=0);


Power over time

All of your answer should be formatted as sentences
For example:
The power on day 0 is VALUE UNIT
Pay attention to the requested output units
Do not pick the complex roots!
1 - What was the power output on Day 0?

In [ ]:
fitted_polynomial(0)     


In [ ]:
f"The power on day 0 is {fitted_polynomial(0):.1f} {my_y.unit}" 


2 - How many YEARS after landing could you still power a 60 W lightbulb?

In [ ]:
my_solution2 = (fitted_polynomial - 60).roots
real_solution2 = my_solution2[np.isreal(my_solution2)].real

my_year2 = (real_solution2[0] * my_x.unit).to(u.yr)


In [ ]:
f"Up to {my_year2.value:.1f}{my_year2.unit} after landing"


3 - How many YEARS after landing could you still power a 5 W USB device?

In [ ]:
my_solution3 = (fitted_polynomial - 5).roots
real_solution3 = my_solution3[np.isreal(my_solution3)].real

my_year3 = (real_solution3[0] * my_x.unit).to(u.yr)


In [ ]:
f"Up to {my_year3.value:.1f}{my_year3.unit} after landing"


4 - How many YEARS after landing until the power output is 0 W?

In [ ]:
my_solution4 = fitted_polynomial.roots
real_solution4 = my_solution4[np.isreal(my_solution4)].real

my_year4 = (real_solution4[0] * my_x.unit).to(u.yr)


In [ ]:
f"Up to {my_year4.value:.1f}{my_year4.unit} after landing"


Fitting data to a function

The datafile
https://uwashington-astro300.github.io/A300_Data/linedata.csv
contains two columns of data [no units]

Read in the Data as an astropy
Qtable

In [ ]:
my_data2 = QTable.read('https://uwashington-astro300.github.io/A300_Data/linedata.csv', 
                       format='ascii.csv')


Plot the Data

Output size w:11in, h:8.5in
Make the plot look nice (including clear labels and a legend)

In [ ]:
my_x2 = my_data2['wavelength']
my_y2 = my_data2['flux']

fig, ax = plt.subplots(
    figsize = (11, 8.5), 
    constrained_layout = True
)

ax.set_xlabel(f"Wavelength")
ax.set_ylabel(f"Flux")

ax.plot(my_x2, my_y2,
        color = "m",
        marker = "o",
        linestyle = "None",
        markersize = 10,
        label = "Data");

ax.legend(loc=0);


Fit a gaussian of the form:

$$ \huge f(x) = A e^{-\frac{(x - C)^2}{W}} $$
A = amplitude of the gaussian
C = x-value of the central peak of the gaussian
W = width of the gaussian
Find the values
(A,C,W)
that best fit the data

In [ ]:
def gauss(x, A, C, W):
    return A * np.exp(-(x - C)**2 / (W**2))


In [ ]:
my_guess_A=my_y2.max()
my_guess_C=my_x2[np.argmax(my_y2)]
my_guess_W=(my_x2.max()-my_x2.min())/5

init_guess=[my_guess_A, my_guess_C, my_guess_W]

fitpars, error = curve_fit(gauss, my_x2, my_y2, p0=init_guess)

A, C, W = fitpars

print(fitpars)
f"The values of A, C, and W are {A}, {C}, and {W} after landing"


Plot the Data and the Fit on the same plot

Output size w:11in, h:8.5in
Make the plot look nice (including clear labels and a legend)

In [ ]:
my_x2 = my_data2['wavelength']
my_y2 = my_data2['flux']

fig, ax = plt.subplots(
    figsize = (11, 8.5), 
    constrained_layout = True
)

ax.set_xlabel(f"Wavelength")
ax.set_ylabel(f"Flux")

ax.plot(my_x2, my_y2,
        color = "m",
        marker = "o",
        linestyle = "None",
        markersize = 10,
        label = "Data");

ax.plot(my_x2, gauss(my_x2.value, *fitpars),
        color = (0.3, 0.1, 0.9, 0.4),
        marker = "None",
        linestyle = "-",
        linewidth = 6,
        label = "Fit to Data")

ax.legend(loc=0);


Due Wed Feb 11 - 1 pm

File -> Download as -> HTML (.html)
upload your .html file to the class Canvas page